# 12 — Ghép head OS vào `sdc_iden.onnx`, tạo bản release

Đầu vào: `sdc_iden.onnx` (09); head OS là `os.joblib` của **cùng run train** (05) mà file đó được dựng từ. Đầu ra: `Models/<ts>/release/` — bản deploy, nằm cùng run với `train/` và `iden/`.
Chạy lại 12 trên cùng run thì thay cả `release/` cũ (kể cả TFLite của 13); muốn giữ bản cũ thì train run mới bằng 05.

| File | Nội dung |
|---|---|
| `sdc_os.onnx` | `input` `string [N,12]` (+ `thresholds` tuỳ chọn) → `output` `string [N,3]` make/type/model sau cascade OS |
| `sdc_os_contract.json` | bản sao contract nhúng trong file |
| `manifest.json` | run nguồn (train, iden), sha256 và kết quả kiểm của từng file; **13** thêm `sdc_os.tflite` vào đây |

**Ghép chứ không dựng lại.** File `sdc_iden.onnx` đang deploy được nạp nguyên vẹn; chỉ node `Concat` ra
`output` bị thay. Head OS đọc lại các tensor graph đã tính sẵn (12 cột, 31 cờ số, token PRL), cascade đọc chỉ
số nhãn cuối của từng head (`sdc/final/<head>`), rồi tra nhãn và nối thành `output` như cũ. Phần iden vì vậy
giống bản deploy tới từng node; ngưỡng iden vẫn ghi đè được qua input `thresholds`.

**Head OS trong ONNX:**

* encoder OS: `ColumnTransformer` của bundle OS qua skl2onnx (như 09 làm với encoder iden), `mac_is_random`
  suy từ `mac_oui` (`mac_random_from_oui`);
* rừng OS: duyệt cây bằng `Gather`/`LessOrEqual`/`Where` trên mảng phẳng của `flatten_forest`, xác suất là trung
  bình float32 — **không** dùng `TreeEnsembleClassifier`, vì cộng dồn trọng số `1/300` đã làm tròn có thể đẩy
  một cửa sổ đúng `0.6` (180/300 cây) xuống dưới ngưỡng;
* luật `HOST_OS` / VCI: ONNX không có op `strip`/`lower`/thay thế chuỗi, nên `re.search(p, s.strip().lower())`
  được viết lại thành một `RegexFullMatch` trên chuỗi thô: khoảng trắng hai đầu nuốt bằng lớp ký tự khoảng
  trắng của Python, chữ hoa/thường bằng `(?i)`;
* chữ ký PRL: dùng token `dhcp_prl` tách theo `,` và `dhcp_prl_len` mà graph đã có — đúng với PRL dạng chuẩn
  mà contract 09 đã bắt buộc (thập phân, cách bởi `,`, không số 0 đầu). Bản TFLite (13) theo đúng quy ước này.

Chạy được trên kernel thường của project (cần `onnx`, `onnxruntime`, `skl2onnx`; mục 5 cần `pyarrow`) — không cần TensorFlow.

In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


## Thư viện

Hàm và hằng dùng chung, chép **nguyên văn** (không còn notebook `00_common`). Notebook khác có bản sao của cùng hàm — sửa ở đây thì sửa cả các bản kia cho khớp: bảng *Hàm có bản sao ở đâu* trong `README.md`. Phần riêng của notebook này bắt đầu sau mục này.

In [2]:
# --- 1. Nền: đường dẫn, lược đồ, mã hoá ---
# Xác định gốc dự án từ thư mục làm việc của kernel Jupyter.
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Code").is_dir() and (p / "Data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Hãy mở notebook từ bên trong dự án SDC")
print("Gốc dự án:", ROOT)

Gốc dự án: D:\01.AI_Security\02.SDC


In [3]:
# --- Đường dẫn dự án ---
from pathlib import Path


DATA = ROOT / "Data"
FEAT_DIR = DATA / "features"
MODELS = ROOT / "Models"
SESSIONS_PATH = DATA / "sessions_verified.parquet"

In [4]:
# --- Lược đồ dữ liệu — head, nhóm feature, chia tập ---
import pandas as pd


MISSING = "<missing>"

META_COLS = ["canonical_device", "scenario", "capture_file", "date", "mac", "n_sources"]

# Ba head, đúng thứ tự output của sản phẩm: hãng / loại / model sản phẩm.
#
# Head thứ ba từng tên là `family` (contract ≤ 2.1). Đổi thành `model` cho khớp ô MODEL
# trong spec, nhưng chỉ là ĐỔI TÊN — nội dung nhãn không đổi, và độ mịn của nó vẫn là
# "model sản phẩm ở mức phân biệt được bằng telemetry": `Amazon Echo Dot` là một model,
# còn `Tuya Plug` vẫn gộp 13 thiết bị ODM không tách được bằng DHCP/DNS/TLS. Đổi tên
# không làm chúng tách ra được; xem `Data/device_labels_verified.csv` cột review_note.
MODEL_HEAD = "model"
LEGACY_MODEL_HEAD = "family"
HEAD_ALIASES = {LEGACY_MODEL_HEAD: MODEL_HEAD}
LABEL_COLS = ["make", "type", MODEL_HEAD]
SOURCES = ["dhcp", "dns", "mdns", "tls"]

# Cột dạng chuỗi token -> TF-IDF. Phần còn lại dạng chuỗi -> categorical nguyên giá trị.
TEXT_COLS = ["dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens"]

# Feature biến mất khi DNS được mã hoá (DoH/DoT) và SNI bị ẩn (ECH). Đây là lý do thật
# để đo bản không có chúng — không phải vì sợ leak: lúc chạy thật router vẫn đọc được
# DNS query và SNI, nên chúng là bằng chứng hợp lệ.
ENCRYPTED_RISK = ["dns_tokens", "tls_sni_tokens"]

# Field driver KHÔNG cung cấp — không feature nào được suy ra từ chúng
FORBIDDEN = ["src_ip", "dst_ip", "dst_port", "ts", "iat", "pktsize", "bytes", "rate"]


# Cot co-mat cua tung nguon. Suy tu `SOURCES` de danh sach nguon chi co MOT ban.
SOURCE_FLAGS = tuple(f"has_{source}" for source in SOURCES)


def feature_sets(df):
    """Các nhóm feature, suy từ chính các cột của dataframe."""
    feats = [c for c in df.columns if c not in META_COLS + LABEL_COLS]
    text = [c for c in TEXT_COLS if c in feats]
    cat = [c for c in feats if c not in text and not pd.api.types.is_numeric_dtype(df[c])]
    num = [c for c in feats if c not in text and c not in cat]
    return {
        "all": feats,
        "text": text,
        "cat": cat,
        "num": num,
        "source": {s: [c for c in feats if c.startswith(s)] for s in SOURCES},
        "no_content": [c for c in feats if c not in ENCRYPTED_RISK],
    }


def assert_serveable(cols):
    """Fail nếu có feature vượt quá telemetry driver cung cấp."""
    bad = [c for c in cols if any(f in c for f in FORBIDDEN)]
    assert not bad, f"Feature không phục vụ được lúc chạy thật: {bad}"
    orphan = [c for c in cols if not c.startswith(("dhcp_", "dns_", "mdns_", "tls_", "has_"))]
    assert not orphan, f"Feature không truy được về nguồn nào: {orphan}"
    return True


def normalize_heads(df):
    """Đổi tên cột nhãn cũ sang tên head hiện hành (`family` -> `model`).

    Làm ở đây, một chỗ, thay vì ghi lại parquet: notebook 03_build_dataset và các file dữ liệu đã sinh
    ra từ trước vẫn dùng được, mà phần còn lại của code chỉ biết đúng một tên. Cột mới
    đã có sẵn thì không đụng vào — dữ liệu mới luôn thắng.
    """
    rename = {old: new for old, new in HEAD_ALIASES.items()
              if old in df.columns and new not in df.columns}
    # Có cả hai cột: cột mới là nhãn đang dùng, cột cũ là tàn dư của file dựng trước lần
    # đổi tên. Bỏ hẳn cột cũ — để lại thì nó không phải nhãn mà cũng không phải feature,
    # và `assert_serveable` sẽ bắt nó như một feature không truy được về nguồn nào.
    stale = [old for old, new in HEAD_ALIASES.items()
             if old in df.columns and new in df.columns]
    if rename or stale:
        df = df.rename(columns=rename).drop(columns=stale)
    return df


def load_sessions(path=SESSIONS_PATH):
    """Nạp feature table mức session kèm nhóm feature đã kiểm tra."""
    assert path.exists(), f"Không tìm thấy {path} — chạy 03_build_dataset.ipynb trước"
    df = normalize_heads(pd.read_parquet(path))
    fs = feature_sets(df)
    assert_serveable(fs["all"])
    return df, fs

In [5]:
# --- Bộ phân lớp L2 ---
from sklearn.ensemble import RandomForestClassifier

# Số cây của RandomForest. Một định nghĩa cho cả 03 (train), 04 (hiệu chỉnh ngưỡng) và
# 05 (xuất ONNX) — ba chỗ này BẮT BUỘC dùng chung một giá trị, vì ngưỡng abstain hiệu
# chỉnh cho model 250 cây không đúng với model 25 cây.
#
# Đo được (CV 5 fold theo ngày): 25 / 100 / 250 cây cho macro-F1 như nhau trong biên độ
# nhiễu — `type` lần lượt 0.8442 / 0.8443 / 0.8440, và 50 cây còn ra 0.8331, tức không
# đơn điệu. Kích thước thì khác hẳn: 1.54 / 6.07 / 15.80 MB.
#
# ⚠️ Số cây ảnh hưởng tới ĐỘ PHÂN GIẢI của confidence mạnh hơn tới accuracy: 25 cây chỉ
# sinh được 26 mức xác suất (bội của 1/25), nên đường coverage–accuracy thành bậc thang
# thô và ngưỡng chọn được kém tinh. Với kiến trúc này, confidence là thứ quyết định phát
# hiện thiết bị lạ, nên đổi số cây thì phải đo lại `ood_abstain` chứ đừng chỉ nhìn F1.
N_ESTIMATORS = 250

RNG = 42


def make_model() -> RandomForestClassifier:
    """Bo phan lop chuan cua du an. Train va danh gia BAT BUOC goi cung ham nay."""
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RNG,
    )

In [6]:
# --- Bộ mã hoá feature (TF-IDF + ordinal) ---
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OrdinalEncoder


def prepare_frame(df, encoder):
    """Ép kiểu đúng như lúc fit: numeric float32, categorical/text chuỗi.

    ColumnTransformer khớp cột theo TÊN nên thứ tự cột đầu vào không quan trọng, nhưng
    kiểu thì có: TF-IDF nhận object/str, để lọt NaN vào là vỡ.
    """
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    out = df[num + cat + text].copy()
    for c in num:
        out[c] = out[c].astype(np.float32)
    for c in cat + text:
        out[c] = out[c].astype(str)
    return out


def encoder_vectorizers(encoder):
    """TF-IDF đã fit, theo cột — 05 cần để ghi tham số vào contract."""
    named = encoder["ct"].named_transformers_
    return {c: named[f"t_{c}"] for c in encoder["text_cols"]}


def apply_encoder(df, encoder):
    """Trả (X, feature_names, cat_index). cat_index dùng cho HistGradientBoosting."""
    num, cat, text = encoder["num_cols"], encoder["cat_cols"], encoder["text_cols"]
    X = encoder["ct"].transform(prepare_frame(df, encoder)).astype(np.float32)
    names = list(num) + list(cat)
    vecs = encoder_vectorizers(encoder)
    for c in text:
        names += [f"{c}::{t}" for t in vecs[c].get_feature_names_out()]
    cat_index = list(range(len(num), len(num) + len(cat)))
    return X, names, cat_index

In [7]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
import hashlib
import re
import struct
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


try:  # Packet parsing is not needed by record-only production deployments.
    from scapy.all import BOOTP, DHCP, DNS, Ether, IP, IPv6, PcapReader, Raw, TCP, UDP
except ImportError:  # pragma: no cover - exercised only in minimal production images
    BOOTP = DHCP = DNS = Ether = IP = IPv6 = PcapReader = Raw = TCP = UDP = None


MAX_TOKENS = 40
TOKEN_SPLIT = re.compile(r"[.\-]")

RAW_FIELDS = {
    "dhcp": ("opt55", "opt60"),
    "dns": ("qname",),
    "mdns": ("qname",),
    "tls": ("version", "ciphers", "alpn", "sni"),
}
# Cột cờ có-mặt nằm ở `sdc_schema` vì đó là lược đồ chung, không riêng của bước parse.
# Gác này bắt trường hợp hai nơi kể tên nguồn lệch nhau — im lặng thì `n_sources` sai.
assert tuple(RAW_FIELDS) == tuple(SOURCES), (
    f"RAW_FIELDS {tuple(RAW_FIELDS)} lệch SOURCES {tuple(SOURCES)}"
)


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
        return bool(missing) if isinstance(missing, (bool, np.bool_)) else False
    except (TypeError, ValueError):
        return False


def _text(value, *, lowercase=False):
    if _is_missing(value):
        return None
    if isinstance(value, bytes):
        value = value.decode(errors="replace")
    value = str(value).strip()
    if not value:
        return None
    return value.lower() if lowercase else value

In [8]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def is_grease(value):
    """RFC 8701 GREASE values have both bytes equal and a low nibble of 0xA."""
    try:
        value = int(value)
    except (TypeError, ValueError):
        return False
    return 0 <= value <= 0xFFFF and (value & 0x0F0F) == 0x0A0A


def normalize_cipher_suites(value):
    if _is_missing(value):
        return MISSING
    values = value if isinstance(value, (list, tuple)) else re.findall(r"\d+", str(value))
    normalized = []
    for item in values:
        try:
            number = int(item)
        except (TypeError, ValueError):
            continue
        if not is_grease(number):
            normalized.append(str(number))
    return ",".join(normalized) if normalized else MISSING


def normalize_tls_version(value):
    if _is_missing(value):
        return MISSING
    text = str(value).strip().lower()
    if text.startswith("0x"):
        text = text[2:]
    if re.fullmatch(r"\d{1,4}", text):
        return text.zfill(4)
    return MISSING

In [9]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def domain_tokens(names, max_tokens=MAX_TOKENS):
    """Domain names to unique lowercase tokens, preserving first-seen order."""
    tokens = []
    for name in names:
        if _is_missing(name):
            continue
        normalized = str(name).lower().rstrip(".")
        tokens.extend(token for token in TOKEN_SPLIT.split(normalized) if token)
    unique = list(dict.fromkeys(tokens))[:max_tokens]
    return " ".join(unique) if unique else MISSING


def _mode(values):
    clean = [str(value) for value in values
             if not _is_missing(value) and str(value).strip() not in ("", MISSING)]
    if not clean:
        return MISSING
    return pd.Series(clean).mode().iat[0]


def _tls_fingerprint(record):
    version = normalize_tls_version(record.get("version", record.get("tls_version")))
    ciphers = normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
    alpn = _text(record.get("alpn"), lowercase=True) or ""
    if MISSING in (version, ciphers):
        return MISSING
    return f"{version}|{ciphers}|{alpn}"

In [10]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def aggregate(records, feature_cols):
    """Aggregate canonical records for one device/window into one model feature row."""
    by_source = {source: [] for source in RAW_FIELDS}
    for record in records:
        if not isinstance(record, dict):
            raise TypeError("Each raw record must be a dictionary")
        proto = record.get("proto")
        if proto not in by_source:
            raise ValueError(f"Invalid proto {proto!r}; expected one of {sorted(RAW_FIELDS)}")
        by_source[proto].append(record)

    row = {}
    row["dhcp_prl"] = _mode(record.get("opt55", record.get("param_req_list"))
                             for record in by_source["dhcp"])
    row["dhcp_vci"] = _mode(record.get("opt60", record.get("vendor_class_id"))
                             for record in by_source["dhcp"])
    options = set()
    if row["dhcp_prl"] != MISSING:
        options = {int(value) for value in re.findall(r"\d+", row["dhcp_prl"])}
    for column in feature_cols:
        if column.startswith("dhcp_opt_"):
            row[column] = int(int(column.rsplit("_", 1)[1]) in options)
    row["dhcp_prl_len"] = len(options)

    row["dns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                         for record in by_source["dns"])
    row["mdns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                          for record in by_source["mdns"])

    tls_records = by_source["tls"]
    row["tls_fp"] = _mode(_tls_fingerprint(record) for record in tls_records)
    row["tls_version"] = _mode(
        normalize_tls_version(record.get("version", record.get("tls_version")))
        for record in tls_records
    )
    row["tls_alpn"] = _mode(_text(record.get("alpn"), lowercase=True)
                              for record in tls_records)
    row["tls_ciphers"] = _mode(
        normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
        for record in tls_records
    )
    row["tls_sni_tokens"] = domain_tokens(record.get("sni") for record in tls_records)

    for source in RAW_FIELDS:
        row[f"has_{source}"] = int(bool(by_source[source]))
    row["n_sources"] = sum(row[flag] for flag in SOURCE_FLAGS)

    numeric_prefixes = ("has_", "dhcp_opt_")
    for column in feature_cols:
        if column not in row:
            row[column] = (0 if column.startswith(numeric_prefixes)
                           or column == "dhcp_prl_len" else MISSING)
    return row

In [11]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
# Hằng của contract `sdc_iden.onnx` (09). 12/13 ghép head OS vào chính graph này nên dùng chung.
import hashlib

IDEN_FORMAT = "sdc-iden-onnx-v2"

ONNX_OPSET = 20          # StringConcat / RegexFullMatch xuất hiện ở opset 20
IR_VERSION = 10
ONNX_LOCALE = "C"        # thiếu locale -> onnxruntime dựng en_US.UTF-8 và chết trên musl
UNKNOWN_LABEL = "__unknown__"
NOT_PRESENT = ("", MISSING, "nan", "None")

HEADS = ("make", "type", "model")

IN_FEATURES = "input"

# Output: đúng MỘT tensor `string [N,3]`, cột theo thứ tự HEADS.
#     output[:, 0] = make   "Generic Laptop"
#     output[:, 1] = type   "Laptop"
#     output[:, 2] = model  "Windows Laptop HP"
# Không trả confidence / topk / retrieval. Xác suất, thứ hạng và tầng nào trả lời đều
# chỉ tồn tại BÊN TRONG graph.
OUT_LABELS = "output"

# Input: 12 cột chuỗi, đúng thứ tự này. Không cột số nào cả — 31 cột số mà encoder cần
# (`has_*`, `dhcp_opt_*`, `dhcp_prl_len`) do graph tự tính từ các cột này, xem
# `build_derived`. Caller không còn chỗ nào để tính cờ lệch với lúc train.
ENCODER_STRINGS = ("dhcp_prl", "tls_fp", "tls_version", "tls_alpn", "tls_ciphers",
                   "dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens")
L0_COLUMNS = ("mac_oui", "dhcp_hostname", "mdns_model")
INPUT_COLUMNS = ENCODER_STRINGS + L0_COLUMNS
# Nguồn có mặt <=> một trong các cột chuỗi của nó có giá trị. `aggregate()` đặt cờ theo
# việc CÓ BẢN GHI, nên hai cách chỉ lệch khi có bản ghi mà không trích được gì — đo ở
# cell "Cờ tự tính so với cờ gốc".
SOURCE_EVIDENCE = {
    # KHÔNG gồm `dhcp_hostname`: agent router lấp cột đó bằng tên từ mDNS/NetBIOS khi cửa
    # sổ không có gói DHCP (`Code/router/hostname_sources.py`). Tính nó vào đây là bật
    # `has_dhcp` cho cửa sổ không có DHCP — đúng điều file đó cấm.
    "has_dhcp": ("dhcp_prl", "dhcp_vci"),
    "has_dns": ("dns_tokens",),
    "has_mdns": ("mdns_tokens",),
    "has_tls": ("tls_fp",),
}
# PRL phải ở dạng chuẩn của extractor: số thập phân cách nhau bởi dấu phẩy, không khoảng
# trắng, không số 0 đầu (`1,3,6,15`). `aggregate()` lấy mọi dãy chữ số bằng regex, còn
# graph không có findall nên tách theo dấu phẩy và so chuỗi — hai cách chỉ trùng nhau
# trên dạng chuẩn.
PRL_SEP = ","

In [12]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
def pick_run(run=None, needs="train/iden.joblib"):
    """Thư mục run `Models/<ts>/`: `run` = tên thư mục, None = run mới nhất có file `needs`.

    Một run gom đầu ra của cả pipeline: `train/` (05), `iden/` (09), `release/` (12, 13). Các bước 09, 12, 13
    chọn run bằng biến `RUN = None` đặt ở đầu mỗi notebook, nên mở notebook nào chạy riêng cũng biết nó
    đang đọc run nào.
    """
    if run:
        path = MODELS / run
        assert (path / needs).is_file(), f"{path} không có {needs}"
        return path
    runs = sorted(p for p in MODELS.iterdir() if (p / needs).is_file())
    assert runs, f"Không thấy run nào có {needs} trong {MODELS}"
    return runs[-1]


# Bản deploy: 12 ghi `Models/<ts>/release/` của run (chạy lại 12 thì thay cả thư mục); 13 thêm bản TFLite
# vào đúng thư mục đó. `manifest.json` ghi run nguồn và sha256 của từng file.
RELEASE_ONNX = "sdc_os.onnx"
RELEASE_MANIFEST = "manifest.json"

In [13]:
# --- Contract `sdc_iden` — input 12 cột chuỗi ---
# Chuẩn bị input 12 cột — bản Python của phần graph tự tính (`build_derived` ở 09).
import re

import numpy as np
import pandas as pd


_MAC_RE = re.compile(r"^[0-9a-f]{2}(:[0-9a-f]{2}){5}$")


def canonical_mac(value):
    if value is None:
        return MISSING
    text = str(value).strip().lower().replace("-", ":")
    return text if _MAC_RE.match(text) else MISSING


def mac_is_random(value):
    """Bit U/L (0x02) của octet đầu: 1 = locally administered = MAC ngẫu nhiên hoá."""
    mac = canonical_mac(value)
    return 0 if mac == MISSING else int(int(mac[:2], 16) & 0x02 != 0)


def mac_oui(value):
    """3 octet đầu. MAC ngẫu nhiên hoá trả <missing> — OUI của nó vô nghĩa."""
    mac = canonical_mac(value)
    if mac == MISSING or mac_is_random(mac):
        return MISSING
    return mac[:8]


def _clean_text(value):
    if value is None:
        return MISSING
    text = str(value).strip().strip(".")
    return text if text and text not in NOT_PRESENT else MISSING


def add_l0_columns(frame):
    """Bổ sung 4 cột L0. Cột nào extractor chưa sinh thì <missing> — graph tra trượt."""
    out = frame.copy()
    macs = out["mac"] if "mac" in out.columns else pd.Series(MISSING, index=out.index)
    out["mac_oui"] = [mac_oui(v) for v in macs]
    out["mac_is_random"] = [mac_is_random(v) for v in macs]
    for col in ("dhcp_hostname", "mdns_model"):
        out[col] = [_clean_text(v) for v in out[col]] if col in out.columns else MISSING
    return out


def prl_options(value):
    """Tập token chữ số của PRL — bản python của phần `StringSplit` trong `build_derived`."""
    text = str(value)
    if text in NOT_PRESENT:
        return set()
    return {t for t in text.split(PRL_SEP) if re.fullmatch(r"[0-9]+", t)}


def derive_numeric(frame, num_cols):
    """Cột số encoder cần + `SOURCE_FLAGS`, tính lại từ cột chuỗi — khớp `build_derived`.

    "Có giá trị" là so khớp chính xác với `NOT_PRESENT`, không strip, vì graph cũng chỉ so
    `Equal` từng chuỗi.
    """
    out = pd.DataFrame(index=frame.index)
    for flag, sources in SOURCE_EVIDENCE.items():
        hit = np.zeros(len(frame), dtype=bool)
        for col in sources:
            hit |= ~frame[col].astype(str).isin(NOT_PRESENT).to_numpy()
        out[flag] = hit.astype(np.float32)
    options = [prl_options(v) for v in frame["dhcp_prl"]]
    for col in num_cols:
        if col.startswith("dhcp_opt_"):
            code = col.rsplit("_", 1)[1]
            out[col] = np.float32([code in o for o in options])
    out["dhcp_prl_len"] = np.float32([len(o) for o in options])
    unknown = [c for c in num_cols if c not in out.columns]
    if unknown:
        raise KeyError(f"không suy được cột số từ chuỗi: {unknown}")
    return out


def matrix_frame(matrix, cols, num_cols):
    """Ma trận chuỗi graph nhận -> frame đủ cột cho encoder, cột số tự tính như graph."""
    frame = pd.DataFrame(matrix, columns=list(cols)).astype(str)
    return pd.concat([frame, derive_numeric(frame, num_cols)], axis=1)


def to_input_matrix(frame, columns):
    """frame -> object array [N, len(columns)] toàn chuỗi, đúng thứ tự contract."""
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(f"thiếu cột: {missing}")
    out = frame[list(columns)].copy()
    for col in columns:
        v = out[col]
        out[col] = v.astype(np.float32).astype(str) if pd.api.types.is_numeric_dtype(v) else v.astype(str)
    return out.to_numpy(dtype=object)


OUI_RE = re.compile(r"[0-9a-fA-F]{2}:[0-9a-fA-F]{2}:[0-9a-fA-F]{2}")


def mac_random_from_oui(value):
    """`mac_is_random` suy từ cột `mac_oui` của contract 12 cột — quy ước của head OS trong 12/13.

    Router luôn biết MAC; extractor chỉ gửi `<missing>` khi MAC ngẫu nhiên hoá. OUI gửi nguyên thì xét
    bit U/L (0x02) của octet đầu.
    """
    v = str(value)
    if v in NOT_PRESENT:
        return 1                              # extractor chỉ trả <missing> khi MAC ngẫu nhiên hoá
    return int(bool(OUI_RE.fullmatch(v)) and int(v[:2], 16) & 0x02 != 0)


def capture_frame():
    """Bộ capture tự thu (`Data/features/*_capture.csv`) ở dạng contract 12 cột."""
    feat = DATA / "features"
    sess = pd.read_csv(feat / "session_features_capture.csv")
    dhcp = pd.read_csv(feat / "dhcp_features_capture.csv")
    hostnames = dhcp.dropna(subset=["hostname"]).drop_duplicates("mac").set_index("mac")["hostname"]
    sess["dhcp_hostname"] = sess["mac"].map(hostnames).fillna(MISSING)
    sess["mdns_model"] = MISSING
    return add_l0_columns(sess)

In [14]:
# --- Khối dựng graph ONNX ---
import numpy as np
import onnx
from onnx import TensorProto as TP
from onnx import helper


class GB:
    """Gom node + initializer cho một graph, tự sinh tên không đụng nhau."""

    def __init__(self, prefix="sdc"):
        self.prefix, self.nodes, self.inits, self._n = prefix, [], [], 0

    def name(self, stem):
        self._n += 1
        return f"{self.prefix}/{stem}_{self._n}"

    def add(self, op_type, inputs, out_stem, **attrs):
        out = self.name(out_stem)
        domain = attrs.pop("domain", "")
        self.nodes.append(helper.make_node(op_type, list(inputs), [out],
                                           name=self.name(f"n_{op_type}"),
                                           domain=domain, **attrs))
        return out

    def add_multi(self, op_type, inputs, out_stems, **attrs):
        outs = [self.name(s) for s in out_stems]
        domain = attrs.pop("domain", "")
        self.nodes.append(helper.make_node(op_type, list(inputs), outs,
                                           name=self.name(f"n_{op_type}"),
                                           domain=domain, **attrs))
        return outs

    def const(self, array, stem, dtype=None):
        out = self.name(stem)
        if dtype == TP.STRING:
            values = np.asarray(array, dtype=object)
            self.inits.append(helper.make_tensor(out, TP.STRING, list(values.shape),
                                                 [s.encode("utf-8") for s in values.ravel()]))
        else:
            self.inits.append(onnx.numpy_helper.from_array(np.asarray(array), out))
        return out


def relink(graph, rename):
    nodes = []
    for node in graph.node:
        copy = onnx.NodeProto()
        copy.CopyFrom(node)
        for i, name in enumerate(copy.input):
            if name in rename:
                copy.input[i] = rename[name]
        nodes.append(copy)
    return nodes, list(graph.initializer)


def str_present(gb, tensor, stem):
    """bool [N,1]: giá trị là chuỗi thật, không phải '', '<missing>', 'nan', 'None'."""
    bad = None
    for token in NOT_PRESENT:
        eq = gb.add("Equal", [tensor, gb.const(np.array([[token]]), f"tok/{stem}", TP.STRING)],
                    f"eq/{stem}")
        bad = eq if bad is None else gb.add("Or", [bad, eq], f"or/{stem}")
    return gb.add("Not", [bad], f"present/{stem}")


def float_eq(gb, tensor, value, stem):
    return gb.add("Equal", [tensor, gb.const(np.float32([[value]]), f"c/{stem}")], f"eq/{stem}")


def set_locale(model, locale=ONNX_LOCALE):
    patched = []
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        for existing in [a for a in node.attribute if a.name == "locale"]:
            node.attribute.remove(existing)
        node.attribute.append(helper.make_attribute("locale", locale))
        patched.append(node.name)
    return patched


def assert_locale(model, locale=ONNX_LOCALE):
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        got = {a.name: a for a in node.attribute}.get("locale")
        assert got is not None and got.s.decode() == locale, f"{node.name}: locale sai"

In [15]:
# --- Head OS (05 phần B) ---
import re


OS_CLASSES = ["iOS", "Android", "Windows", "macOS", "Linux", "Embedded"]
OS_THRESHOLD = 0.6          # ngưỡng của rừng (L2); L0/L1 là luật, conf = 1.0


def prl_set(prl):
    text = str(prl)
    return frozenset() if text in NOT_PRESENT else frozenset(int(t) for t in re.findall(r"\d+", text))


HOST_OS = [
    (r"^(iphone|ipad|ipod)\b", "iOS"),
    (r"^(macbook|imac|mac-?mini|mac-?pro|mac-?studio)\b", "macOS"),
    (r"^(android|galaxy|sm-|redmi|xiaomi|poco|mi-|realme|oppo|vivo|oneplus|pixel|huawei|honor|nokia|moto)", "Android"),
    (r"^(desktop|laptop|win)-[a-z0-9]{4,}$", "Windows"),
    (r"^(raspberrypi|ubuntu|debian|fedora|archlinux)\b", "Linux"),
    (r"^(esp[_-]|tuya|shelly|tasmota|wled)", "Embedded"),
]


def l0_os(hostname, vci, mac_random):
    """Chuỗi thiết bị tự khai."""
    h = str(hostname or "").strip().lower()
    if h and h not in ("<missing>", "nan", "none"):
        for pattern, os_ in HOST_OS:
            if re.search(pattern, h):
                return os_, "hostname"
    v = str(vci or "").strip().lower()
    if v.startswith("msft "):
        return "Windows", "vci"
    # android-dhcp-* cũng có ở loa Echo Studio (MAC thật) -> chỉ tin khi MAC ngẫu nhiên.
    if v.startswith("android-dhcp-") and mac_random:
        return "Android", "vci"
    return None, None


def l1_os(signatures, prl, mdns, mac_random, min_jaccard=0.9):
    """Vân tay giao thức: tập option DHCP gần nhất, rồi Apple Continuity qua mDNS."""
    s = prl_set(prl)
    if s:
        best, owners = 0.0, set()
        for key, value in signatures.items():
            j = len(s & key) / len(s | key)
            if j > best + 1e-9:
                best, owners = j, set(value)
            elif abs(j - best) <= 1e-9:
                owners |= value
        if best >= min_jaccard and len(owners) == 1:
            os_ = next(iter(owners))
            # Vân tay thiết bị gia dụng mà MAC ngẫu nhiên hoá: bằng chứng trái nhau, nhường L2.
            if not (os_ == "Embedded" and mac_random):
                return os_, f"prl~{best:.2f}"
    # Handoff/AirDrop: Apple TV/HomePod cũng quảng bá nhưng dùng MAC thật.
    m = str(mdns)
    if mac_random and ("_companion link" in m or "_rdlink" in m):
        return "iOS", "mdns"
    return None, None


def os_decide(bundle, frame, thr=OS_THRESHOLD, use_hostname=True):
    """-> list (nhãn, nguồn, conf) cho từng dòng."""
    X, _, _ = apply_encoder(frame[bundle["cols"]], bundle["encoder"])
    proba = bundle["model"].predict_proba(X)
    classes = bundle["model"].classes_
    out = []
    for i, (_, r) in enumerate(frame.iterrows()):
        rnd = int(r.get("mac_is_random", 0))
        label, src = l0_os(r.get("dhcp_hostname") if use_hostname else None, r.get("dhcp_vci"), rnd)
        if not label:
            label, src = l1_os(bundle["signatures"], r.get("dhcp_prl"), r.get("mdns_tokens"), rnd)
        if label:
            out.append((label, src, 1.0))
            continue
        k = int(proba[i].argmax()); p, conf = classes[k], float(proba[i, k])
        # Vân tay TLS là của TRÌNH DUYỆT (Chrome trên Windows = Chrome trên Linux), thứ tự
        # cipher của Apple xáo theo phiên: cửa sổ chỉ-TLS không đủ để nói một OS đa dụng.
        tls_only = not (r.get("has_dhcp") == 1 or r.get("has_mdns") == 1)
        if conf >= thr and not (p != "Embedded" and tls_only):
            out.append((p, "forest", conf))
        else:
            out.append((UNKNOWN_LABEL, "abstain", conf))
    return out

In [16]:
# --- Head OS (05 phần B) ---
TYPE_OK = {"iOS": {"Smartphone"}, "Android": {"Smartphone"}, "Windows": {"Laptop"},
           "macOS": {"Laptop"}, "Linux": {"Laptop", "Single-board Computer"}}
GENERAL_PURPOSE_TYPES = {"Smartphone", "Laptop", "Single-board Computer"}
MAKE_OK = {"iOS": {"Apple"}, "macOS": {"Apple"}}


def cascade(os_label, mac_random, make, type_, model):
    """-> (os, make, type, model, ghi chú)."""
    notes = []
    if os_label != UNKNOWN_LABEL:
        allowed = TYPE_OK.get(os_label)
        if os_label == "Embedded" and type_ in GENERAL_PURPOSE_TYPES:
            notes.append(f"type {type_} mâu thuẫn Embedded"); type_ = UNKNOWN_LABEL
        elif type_ != UNKNOWN_LABEL and allowed and type_ not in allowed:
            notes.append(f"type {type_} mâu thuẫn {os_label}"); type_ = UNKNOWN_LABEL
        if type_ == UNKNOWN_LABEL:
            if os_label == "iOS" or (os_label == "Android" and mac_random):
                type_ = "Smartphone"; notes.append("type suy từ OS")
            elif os_label in ("Windows", "macOS"):
                type_ = "Laptop"; notes.append("type suy từ OS")
        allowed = MAKE_OK.get(os_label)
        if make != UNKNOWN_LABEL and allowed and make not in allowed:
            notes.append(f"make {make} mâu thuẫn {os_label}"); make = UNKNOWN_LABEL
        if make == "Apple" and os_label not in ("iOS", "macOS"):
            notes.append(f"make Apple mâu thuẫn {os_label}"); make = UNKNOWN_LABEL
        if make == UNKNOWN_LABEL and allowed:
            make = "Apple"; notes.append("make suy từ OS")
    if (make == UNKNOWN_LABEL or type_ == UNKNOWN_LABEL) and model != UNKNOWN_LABEL:
        notes.append("model bỏ vì make/type abstain"); model = UNKNOWN_LABEL
    return os_label, make, type_, model, "; ".join(notes)

In [17]:
# --- Xuất rừng sang mảng phẳng và TFLite (12, 13) ---
import numpy as np


def float32_floor(values):
    """float32 lớn nhất <= values (float64) — giữ nguyên kết quả của phép so x_f32 <= t."""
    out = values.astype(np.float32)
    up = out.astype(np.float64) > values
    out[up] = np.nextafter(out[up], np.float32(-np.inf))
    return out


def flatten_forest(model):
    feature, threshold, left, right, value, roots = [], [], [], [], [], []
    offset = 0
    for est in model.estimators_:
        t = est.tree_
        n = t.node_count
        own = np.arange(n)
        leaf = t.children_left == -1
        # cây con nhận nhãn đã mã hoá 0..C-1 — cột value theo đúng thứ tự `model.classes_`
        assert t.value.shape[1:] == (1, len(model.classes_)), t.value.shape
        feature.append(np.where(leaf, 0, t.feature))
        threshold.append(np.where(leaf, 0.0, float32_floor(t.threshold)))
        left.append(np.where(leaf, own, t.children_left) + offset)
        right.append(np.where(leaf, own, t.children_right) + offset)
        v = t.value[:, 0, :]
        value.append(v / v.sum(axis=1, keepdims=True))
        roots.append(offset)
        offset += n
    return {
        "feature": np.concatenate(feature).astype(np.int32),
        "threshold": np.concatenate(threshold).astype(np.float32),
        "left": np.concatenate(left).astype(np.int32),
        "right": np.concatenate(right).astype(np.int32),
        "value": np.concatenate(value).astype(np.float32),
        "roots": np.asarray(roots, dtype=np.int32),
        "depth": max(est.tree_.max_depth for est in model.estimators_),
    }

In [18]:
from pathlib import Path
import hashlib
import inspect
import itertools
import json
import re
import shutil
import warnings
from datetime import datetime

import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from IPython.display import display
from onnx import TensorProto as TP
from onnx import compose, helper
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

warnings.filterwarnings("ignore", message=".*InconsistentVersion.*")
ort.set_default_logger_severity(3)
print("onnx", onnx.__version__, "| onnxruntime", ort.__version__)

onnx 1.22.0 | onnxruntime 1.28.0


## 1. Chọn run

`RUN` = tên thư mục run `Models/<ts>/`; `None` = run mới nhất đã có `iden/sdc_iden.onnx`. Head OS không chọn riêng: lấy `train/os.joblib` của cùng run, và kiểm `source_run` trong contract của `sdc_iden.onnx` trỏ đúng run đó. Bộ dựng graph `GB`, luật OS, `cascade`, `flatten_forest`… ở mục **Thư viện** đầu notebook.

In [19]:
COLS = list(INPUT_COLUMNS)

RUN = None                        # run Models/<ts>/ đã qua 09; None = mới nhất

run_dir = pick_run(RUN, "iden/sdc_iden.onnx")
iden_dir, train_dir = run_dir / "iden", run_dir / "train"
onnx_path = iden_dir / "sdc_iden.onnx"
base_model = onnx.load(str(onnx_path))
base_contract = json.loads({p.key: p.value for p in base_model.metadata_props}["sdc.contract"])
assert base_contract["format"] == IDEN_FORMAT and base_contract["columns"] == COLS
LABELS = base_contract["labels"]

# head OS lấy từ CÙNG run train mà sdc_iden.onnx này được dựng từ — không thể ghép lệch run
assert base_contract["source_run"] == run_dir.name, \
    f"sdc_iden.onnx dựng từ run {base_contract['source_run']}, không phải {run_dir.name}"
os_bundle = joblib.load(train_dir / "os.joblib")
assert os_bundle["format"] == "sdc-os-v0", os_bundle["format"]
assert OS_THRESHOLD == os_bundle["threshold"] and list(OS_CLASSES) == list(os_bundle["classes"])
MIN_JACCARD = inspect.signature(l1_os).parameters["min_jaccard"].default
OS_LABELS = list(os_bundle["classes"]) + [UNKNOWN_LABEL]
OSI = {c: i for i, c in enumerate(OS_LABELS)}
print(f"run: {run_dir.name}  |  sdc_iden.onnx {onnx_path.stat().st_size / 1024:.0f} KiB"
      f"  |  lớp OS {OS_LABELS}")

train: 20261005_140503_train  |  iden: 20261005_140605_iden (1598 KiB)  |  lớp OS ['iOS', 'Android', 'Windows', 'macOS', 'Linux', 'Embedded', '__unknown__']


## 2. Tensor dùng lại từ graph đang deploy

Tên do `GB` của 09 sinh: `sdc/col/<cột>` (12 cột sau `Split`), `sdc/num/<cờ>` (`has_*`, `dhcp_opt_*`),
`sdc/n/prl_len`, `sdc/prl/tokens` (token `dhcp_prl` tách theo `,`), `sdc/final/<head>` (chỉ số nhãn sau
ngưỡng), bảng nhãn `sdc/lab/<head>`. Mỗi tên phải khớp **đúng một** tensor.

In [20]:
_outputs = [o for n in base_model.graph.node for o in n.output]
_inits = {i.name for i in base_model.graph.initializer}


def one(pattern, pool):
    hits = [o for o in pool if re.fullmatch(pattern, o)]
    assert len(hits) == 1, (pattern, hits)
    return hits[0]


split = next(n for n in base_model.graph.node if n.op_type == "Split" and n.input[0] == IN_FEATURES)
COL = dict(zip(COLS, split.output))
IDEN_NUM = list(base_contract["derived_in_graph"]["columns"])
DERIVED = {c: one(rf"sdc/num/{re.escape(c)}_\d+", _outputs) for c in IDEN_NUM if c != "dhcp_prl_len"}
DERIVED["dhcp_prl_len"] = one(r"sdc/n/prl_len_\d+", _outputs)
PRL_TOKENS = one(r"sdc/prl/tokens_\d+", _outputs)
FINAL = {h: one(rf"sdc/final/{h}_\d+", _outputs) for h in HEADS}
LABEL_TABLE = {h: one(rf"sdc/lab/{h}_\d+", _inits) for h in HEADS}
OLD_OUT = [n for n in base_model.graph.node if OUT_LABELS in n.output]
assert len(OLD_OUT) == 1 and OLD_OUT[0].op_type == "Concat"
print(f"cột {len(COL)}, cờ số {len(DERIVED)}, token PRL {PRL_TOKENS}, chỉ số cuối {FINAL}")

cột 12, cờ số 31, token PRL sdc/prl/tokens_129, chỉ số cuối {'make': 'sdc/final/make_1083', 'type': 'sdc/final/type_1392', 'model': 'sdc/final/model_742'}


## 3. Dựng head OS + cascade và ghép

Mỗi khối theo đúng hàm tương ứng ở mục **Thư viện** đầu notebook (head OS); mục 4 kiểm từng khối với chính hàm đó.

In [21]:
# --- Quy ước và hằng ----------------------------------------------------------------------
# Khoảng trắng mà str.strip() của Python bỏ: \t\n\v\f\r, \x1c-\x1f, \x85 và mọi ký tự nhóm Z.
WS = r"[\t\n\x{0b}\f\r\x{1c}-\x{1f}\x{85}\p{Z}]"
NOT_WS = "[^" + WS[1:]
OUI_BIT = "[0-9a-fA-F][2367abefABEF]:[0-9a-fA-F]{2}:[0-9a-fA-F]{2}"


def host_regex(pattern):
    """`re.search(pattern, raw.strip().lower())` -> một RegexFullMatch (RE2) trên chuỗi thô."""
    assert pattern.startswith("^") and not pattern.endswith(r"\$"), pattern
    q = pattern[1:]
    if q.endswith("$"):
        assert r"\b" not in q, pattern
        return f"(?s){WS}*(?i:{q[:-1]}){WS}*"
    if r"\b" in q:           # \b của Python là Unicode; RE2 thì không -> biên phải viết tường minh
        assert q.endswith(r"\b") and q.count(r"\b") == 1, pattern
        q = q[:-2] + r"(?:[^\p{L}\p{N}_]|$)"
    return f"(?s){WS}*(?i:{q}).*"


HOST_EXCLUDED = f"(?s){WS}*(?i:<missing>|nan|none)?{WS}*"          # h rỗng / <missing> / nan / none
VCI_MSFT = f"(?s){WS}*(?i:msft) .*{NOT_WS}.*"                       # strip().lower().startswith("msft ")
VCI_ANDROID = f"(?s){WS}*(?i:android-dhcp-).*"
MDNS_APPLE = "(?s).*(?:_companion link|_rdlink).*"

FLAT_OS = flatten_forest(os_bundle["model"])
OS_WIDTH = int(os_bundle["model"].n_features_in_)
FOREST_OS = np.int64([OSI[str(c)] for c in os_bundle["model"].classes_])
SIG_KEYS = list(os_bundle["signatures"])
SIG_OPTIONS = sorted({o for k in SIG_KEYS for o in k})
SIG_MEMBER = np.float32([[o in k for k in SIG_KEYS] for o in SIG_OPTIONS])                       # [U, K]
SIG_SIZE = np.float32([[len(k) for k in SIG_KEYS]])                                               # [1, K]
SIG_OWNER = np.float32([[[c in os_bundle["signatures"][k] for c in OS_LABELS] for k in SIG_KEYS]])  # [1, K, O]
TYPE_L, MAKE_L, MODEL_L = LABELS["type"], LABELS["make"], LABELS[MODEL_HEAD]


# --- Khối graph ---------------------------------------------------------------------------
def I(gb, v, stem):
    return gb.const(np.int64([[v]]), stem)


def F(gb, v, stem):
    return gb.const(np.float32([[v]]), stem)


def rx(gb, tensor, pattern, stem):
    return gb.add("RegexFullMatch", [tensor], stem, pattern=pattern)


def build_random(gb, oui):
    """bool [N,1]: MAC ngẫu nhiên hoá."""
    absent = gb.add("Not", [str_present(gb, oui, "oui")], "oui/absent")
    return gb.add("Or", [absent, rx(gb, oui, OUI_BIT, "oui/bit")], "random")


def build_os_encoder(gb, source):
    """ColumnTransformer của bundle OS qua skl2onnx -> (nodes, inits, opsets, tensor feature [N, 187])."""
    enc = os_bundle["encoder"]
    initial = ([(c, FloatTensorType([None, 1])) for c in enc["num_cols"]]
               + [(c, StringTensorType([None, 1])) for c in enc["cat_cols"] + enc["text_cols"]])
    ct = convert_sklearn(enc["ct"], "sdc_os_encoder", initial_types=initial, target_opset=ONNX_OPSET)
    ct = compose.add_prefix(ct, "osenc/", rename_inputs=False)
    cols = enc["num_cols"] + enc["cat_cols"] + enc["text_cols"]
    nodes, inits = relink(ct.graph, {i.name: source[c] for i, c in zip(ct.graph.input, cols)})
    return nodes, inits, list(ct.opset_import), ct.graph.output[0].name


def build_forest(gb, x, flat, width, stem):
    """Duyệt rừng phẳng (`flatten_forest`): mỗi vòng đưa mọi cây xuống một tầng; xác suất = trung bình float32."""
    n = gb.add("Gather", [gb.add("Shape", [x], f"{stem}/shape"), gb.const(np.int64(0), f"{stem}/i0")],
               f"{stem}/n", axis=0)
    rows = gb.add("Range", [gb.const(np.int64(0), f"{stem}/r0"), n, gb.const(np.int64(1), f"{stem}/r1")],
                  f"{stem}/range")
    row = gb.add("Mul", [gb.add("Unsqueeze", [rows, gb.const(np.int64([1]), f"{stem}/ax1")], f"{stem}/rowu"),
                         gb.const(np.int64(width), f"{stem}/w")], f"{stem}/row")                  # [N, 1]
    shape = gb.add("Concat", [gb.add("Unsqueeze", [n, gb.const(np.int64([0]), f"{stem}/ax0")], f"{stem}/n1"),
                              gb.const(np.int64([len(flat["roots"])]), f"{stem}/t")], f"{stem}/nt", axis=0)
    node = gb.add("Expand", [gb.const(flat["roots"].astype(np.int64).reshape(1, -1), f"{stem}/roots"), shape],
                  f"{stem}/node")                                                                  # [N, T]
    xf = gb.add("Reshape", [x, gb.const(np.int64([-1]), f"{stem}/flat")], f"{stem}/xf")
    feature = gb.const(flat["feature"].astype(np.int64), f"{stem}/feature")
    threshold = gb.const(flat["threshold"], f"{stem}/threshold")
    left = gb.const(flat["left"].astype(np.int64), f"{stem}/left")
    right = gb.const(flat["right"].astype(np.int64), f"{stem}/right")
    for d in range(int(flat["depth"])):
        idx = gb.add("Add", [gb.add("Gather", [feature, node], f"{stem}/f", axis=0), row], f"{stem}/idx")
        xv = gb.add("Gather", [xf, idx], f"{stem}/xv", axis=0)
        le = gb.add("LessOrEqual", [xv, gb.add("Gather", [threshold, node], f"{stem}/thr", axis=0)], f"{stem}/le")
        node = gb.add("Where", [le, gb.add("Gather", [left, node], f"{stem}/l", axis=0),
                                gb.add("Gather", [right, node], f"{stem}/r", axis=0)], f"{stem}/node")
    values = gb.add("Gather", [gb.const(flat["value"], f"{stem}/value"), node], f"{stem}/values", axis=0)
    return gb.add("ReduceMean", [values, gb.const(np.int64([1]), f"{stem}/axt")], f"{stem}/proba", keepdims=0)


def build_os_head(gb, col, derived, prl_tokens, x_os, random):
    """`os_decide()` (mục Thư viện): L0 hostname -> L0 VCI -> L1 PRL -> L1 mDNS -> rừng. int64 [N,1]."""
    minus = I(gb, -1, "os/m1")
    host = minus
    for pattern, os_ in reversed(HOST_OS):                       # luật đầu tiên khớp thắng
        host = gb.add("Where", [rx(gb, col["dhcp_hostname"], host_regex(pattern), "host/rx"),
                                I(gb, OSI[os_], "host/os"), host], "host")
    host = gb.add("Where", [rx(gb, col["dhcp_hostname"], HOST_EXCLUDED, "host/excl"), minus, host], "host/ok")

    android = gb.add("And", [rx(gb, col["dhcp_vci"], VCI_ANDROID, "vci/and"), random], "vci/and_rnd")
    vci = gb.add("Where", [rx(gb, col["dhcp_vci"], VCI_MSFT, "vci/msft"), I(gb, OSI["Windows"], "vci/win"),
                           gb.add("Where", [android, I(gb, OSI["Android"], "vci/android"), minus], "vci/a")], "vci")

    has = []
    for o in SIG_OPTIONS:
        eq = gb.add("Equal", [prl_tokens, gb.const(np.array([[str(o)]]), f"prl/o{o}", TP.STRING)], f"prl/eq{o}")
        cnt = gb.add("ReduceSum", [gb.add("Cast", [eq], f"prl/f{o}", to=TP.FLOAT),
                                   gb.const(np.int64([1]), f"prl/ax{o}")], f"prl/n{o}", keepdims=1)
        has.append(gb.add("Cast", [gb.add("Greater", [cnt, F(gb, 0.0, f"prl/z{o}")], f"prl/has{o}")],
                          f"prl/hf{o}", to=TP.FLOAT))
    has = gb.add("Concat", has, "prl/has", axis=1)                                             # [N, U]
    inter = gb.add("MatMul", [has, gb.const(SIG_MEMBER, "prl/member")], "prl/inter")            # [N, K]
    n_opt = derived["dhcp_prl_len"]                                                             # [N, 1]
    union = gb.add("Sub", [gb.add("Add", [n_opt, gb.const(SIG_SIZE, "prl/size")], "prl/u0"), inter], "prl/union")
    ratio = gb.add("Div", [inter, union], "prl/ratio")
    best = gb.add("ReduceMax", [ratio, gb.const(np.int64([1]), "prl/axk")], "prl/best", keepdims=1)
    tied = gb.add("Cast", [gb.add("Equal", [ratio, best], "prl/tie")], "prl/tied", to=TP.FLOAT)
    spread = gb.add("Mul", [gb.add("Unsqueeze", [tied, gb.const(np.int64([2]), "prl/ax2")], "prl/tu"),
                            gb.const(SIG_OWNER, "prl/owner")], "prl/spread")                    # [N, K, O]
    owners = gb.add("ReduceMax", [spread, gb.const(np.int64([1]), "prl/axs")], "prl/owners", keepdims=0)
    n_owner = gb.add("ReduceSum", [owners, gb.const(np.int64([1]), "prl/axo")], "prl/nown", keepdims=1)
    prl_os = gb.add("ArgMax", [owners], "prl/os", axis=1, keepdims=1)
    ok = gb.add("And", [gb.add("Greater", [n_opt, F(gb, 0.0, "prl/z")], "prl/any"),
                        gb.add("GreaterOrEqual", [best, F(gb, MIN_JACCARD, "prl/minj")], "prl/j")], "prl/ok1")
    ok = gb.add("And", [ok, gb.add("Equal", [n_owner, F(gb, 1.0, "prl/one")], "prl/uniq")], "prl/ok2")
    emb_rnd = gb.add("And", [gb.add("Equal", [prl_os, I(gb, OSI["Embedded"], "prl/emb")], "prl/isemb"), random],
                     "prl/embrnd")
    prl_ok = gb.add("And", [ok, gb.add("Not", [emb_rnd], "prl/notemb")], "prl/ok")

    mdns_ok = gb.add("And", [random, rx(gb, col["mdns_tokens"], MDNS_APPLE, "mdns/rx")], "mdns/ok")

    p = build_forest(gb, x_os, FLAT_OS, OS_WIDTH, "rf_os")
    pmax = gb.add("ReduceMax", [p, gb.const(np.int64([1]), "rf/ax")], "rf/pmax", keepdims=1)
    k = gb.add("ArgMax", [p], "rf/k", axis=1, keepdims=1)                 # chỉ số đầu khi hoà, như numpy
    forest_os = gb.add("Gather", [gb.const(FOREST_OS, "rf/map"), k], "rf/os", axis=0)
    tls_only = gb.add("Not", [gb.add("Or", [float_eq(gb, derived["has_dhcp"], 1.0, "rf/dhcp"),
                                            float_eq(gb, derived["has_mdns"], 1.0, "rf/mdns")], "rf/src")],
                      "rf/tls_only")
    general = gb.add("And", [gb.add("Not", [gb.add("Equal", [forest_os, I(gb, OSI["Embedded"], "rf/emb")], "rf/isemb")],
                                    "rf/notemb"), tls_only], "rf/general_tls")
    forest_ok = gb.add("And", [gb.add("GreaterOrEqual", [pmax, F(gb, float(os_bundle["threshold"]), "rf/thr")], "rf/conf"),
                               gb.add("Not", [general], "rf/gate")], "rf/ok")

    os_idx = gb.add("Where", [forest_ok, forest_os, I(gb, OSI[UNKNOWN_LABEL], "os/unk")], "os/l2")
    os_idx = gb.add("Where", [mdns_ok, I(gb, OSI["iOS"], "os/ios"), os_idx], "os/mdns")
    os_idx = gb.add("Where", [prl_ok, prl_os, os_idx], "os/prl")
    os_idx = gb.add("Where", [gb.add("GreaterOrEqual", [vci, I(gb, 0, "os/z1")], "os/hv"), vci, os_idx], "os/vci")
    return gb.add("Where", [gb.add("GreaterOrEqual", [host, I(gb, 0, "os/z0")], "os/hh"), host, os_idx], "os")


def build_cascade(gb, os_idx, random, make, type_, model):
    """`cascade()` (mục Thư viện) trên chỉ số nhãn int64 [N,1]."""
    eq = lambda t, v, s: gb.add("Equal", [t, I(gb, v, s)], s + "/eq")
    is_os = lambda *names: (eq(os_idx, OSI[names[0]], f"cas/{names[0]}") if len(names) == 1 else
                            gb.add("Or", [eq(os_idx, OSI[n], f"cas/{n}") for n in names], "cas/or"))
    lookup = lambda table, idx, s: gb.add("Gather", [gb.const(np.asarray(table), s), idx], s + "/g", axis=0)
    pair = lambda table, idx, width, s: lookup(np.asarray(table).ravel(), gb.add(
        "Add", [gb.add("Mul", [os_idx, I(gb, width, s + "/w")], s + "/m"), idx], s + "/i"), s)
    t_unk, m_unk, mo_unk = TYPE_L.index(UNKNOWN_LABEL), MAKE_L.index(UNKNOWN_LABEL), MODEL_L.index(UNKNOWN_LABEL)
    known = gb.add("Not", [eq(os_idx, OSI[UNKNOWN_LABEL], "cas/unk")], "cas/known")

    emb_general = gb.add("And", [is_os("Embedded"), lookup([t in GENERAL_PURPOSE_TYPES for t in TYPE_L],
                                                           type_, "cas/general")], "cas/emb_general")
    type_ok = pair([[t in TYPE_OK.get(o, ()) for t in TYPE_L] for o in OS_LABELS], type_, len(TYPE_L),
                   "cas/type_ok")
    conflict = gb.add("And", [gb.add("And", [gb.add("Not", [eq(type_, t_unk, "cas/tu")], "cas/tk"),
                                             lookup([o in TYPE_OK for o in OS_LABELS], os_idx, "cas/trule")],
                                     "cas/c1"), gb.add("Not", [type_ok], "cas/tbad")], "cas/conflict")
    drop_t = gb.add("And", [known, gb.add("Or", [emb_general, conflict], "cas/c2")], "cas/drop_t")
    type_ = gb.add("Where", [drop_t, I(gb, t_unk, "cas/t_unk"), type_], "cas/type1")
    empty = gb.add("And", [known, eq(type_, t_unk, "cas/te")], "cas/empty")
    phone = gb.add("And", [empty, gb.add("Or", [is_os("iOS"), gb.add("And", [is_os("Android"), random], "cas/ar")],
                                         "cas/ph")], "cas/phone")
    laptop = gb.add("And", [gb.add("And", [empty, gb.add("Not", [phone], "cas/np")], "cas/e2"),
                            is_os("Windows", "macOS")], "cas/laptop")
    type_ = gb.add("Where", [phone, I(gb, TYPE_L.index("Smartphone"), "cas/smart"),
                             gb.add("Where", [laptop, I(gb, TYPE_L.index("Laptop"), "cas/lap"), type_], "cas/t2")],
                   "cas/type")

    make_rule = lookup([o in MAKE_OK for o in OS_LABELS], os_idx, "cas/mrule")
    make_ok = pair([[m in MAKE_OK.get(o, ()) for m in MAKE_L] for o in OS_LABELS], make, len(MAKE_L),
                   "cas/make_ok")
    bad = gb.add("And", [gb.add("And", [gb.add("Not", [eq(make, m_unk, "cas/mu")], "cas/mk"), make_rule], "cas/b1"),
                         gb.add("Not", [make_ok], "cas/mbad")], "cas/bad")
    make = gb.add("Where", [gb.add("And", [known, bad], "cas/kb"), I(gb, m_unk, "cas/m_unk"), make], "cas/make1")
    apple = MAKE_L.index("Apple")
    stray = gb.add("And", [gb.add("And", [known, eq(make, apple, "cas/ap")], "cas/ka"),
                           gb.add("Not", [is_os("iOS", "macOS")], "cas/nonapple")], "cas/stray")
    make = gb.add("Where", [stray, I(gb, m_unk, "cas/m_unk2"), make], "cas/make2")
    fill = gb.add("And", [gb.add("And", [known, eq(make, m_unk, "cas/me")], "cas/f1"), make_rule], "cas/fill")
    make = gb.add("Where", [fill, I(gb, apple, "cas/apple"), make], "cas/make")

    gone = gb.add("Or", [eq(make, m_unk, "cas/mu2"), eq(type_, t_unk, "cas/tu2")], "cas/gone")
    drop_m = gb.add("And", [gone, gb.add("Not", [eq(model, mo_unk, "cas/mou")], "cas/mok")], "cas/drop_m")
    return make, type_, gb.add("Where", [drop_m, I(gb, mo_unk, "cas/mo_unk"), model], "cas/model")


# --- Sắp xếp topo + bỏ node chết -------------------------------------------------------------
def topo_prune(nodes, inits, inputs, outputs):
    """Sắp xếp topo (checker bắt buộc), rồi bỏ node/initializer không dẫn tới output."""
    ready, order, pending = set(inputs) | {i.name for i in inits}, [], list(nodes)
    while pending:
        rest = []
        for n in pending:
            if all((not i) or i in ready for i in n.input):
                order.append(n)
                ready.update(n.output)
            else:
                rest.append(n)
        assert len(rest) < len(pending), "graph có vòng hoặc thiếu tensor"
        pending = rest
    need, keep = set(outputs), []
    for n in reversed(order):
        if any(o in need for o in n.output):
            keep.append(n)
            need.update(i for i in n.input if i)
    keep.reverse()
    used = {i for n in keep for i in n.input}
    return keep, [t for t in inits if t.name in used or t.name in inputs]


# --- Lắp ráp -----------------------------------------------------------------------------
gb = GB("sdcos")
random = build_random(gb, COL["mac_oui"])
source = {**COL, **DERIVED, "mac_is_random": gb.add("Cast", [random], "random_f", to=TP.FLOAT)}
enc_nodes, enc_inits, enc_opsets, x_os = build_os_encoder(gb, source)
os_idx = build_os_head(gb, COL, DERIVED, PRL_TOKENS, x_os, random)
make, type_, model_idx = build_cascade(gb, os_idx, random, FINAL["make"], FINAL["type"], FINAL[MODEL_HEAD])
labels = [gb.add("Gather", [LABEL_TABLE[h], t], f"out/{h}", axis=0)
          for h, t in zip(HEADS, (make, type_, model_idx))]                      # mỗi head string [N,1]
gb.nodes.append(helper.make_node("Concat", labels, [OUT_LABELS], name=gb.name("n_out"), axis=1))

g = base_model.graph
graph_inputs = [i.name for i in g.input]
nodes, inits = topo_prune([n for n in g.node if n is not OLD_OUT[0]] + enc_nodes + gb.nodes,
                          list(g.initializer) + enc_inits + gb.inits, graph_inputs, [OUT_LABELS])
opsets = {o.domain: o.version for o in base_model.opset_import}
for o in enc_opsets:
    opsets[o.domain] = max(opsets.get(o.domain, 0), o.version)
graph = helper.make_graph(nodes, "sdc_iden_os", list(g.input), list(g.output), inits)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid(d, v) for d, v in opsets.items()])
model.ir_version = IR_VERSION
patched = set_locale(model)
assert_locale(model)
onnx.checker.check_model(model)
print(f"node {len(g.node)} -> {len(nodes)}  |  StringNormalizer đặt locale C: {len(patched)}  |  opset {opsets}")

node 633 -> 1080  |  StringNormalizer đặt locale C: 6  |  opset {'': 20, 'ai.onnx.ml': 3, 'com.microsoft': 1}


## 4. Kiểm nhanh

Hai phép kiểm, vài giây, đều so với **đường Python** (`os_decide`, `cascade` ở mục Thư viện):

1. **Cascade** — dựng riêng khối cascade thành một model nhỏ, chạy mọi tổ hợp (OS, MAC ngẫu nhiên, make, type,
   model), so với `cascade()`.
2. **Đầu cuối** — bộ capture và các dòng ép từng tầng OS (mọi luật hostname kể cả hoa/thường, khoảng trắng,
   chữ Việt sát biên `\b`; VCI; PRL quanh từng chữ ký ở dạng chuẩn; mDNS Apple; MAC thiếu / thật / ngẫu nhiên):
   output của file mới so với `cascade(os_decide(...), nhãn của sdc_iden.onnx gốc)`.

In [22]:
# 1. Cascade: mọi tổ hợp
cg = GB("cas")
cin = [helper.make_tensor_value_info(n, TP.INT64 if n != "random" else TP.BOOL, [None, 1])
       for n in ("os", "random", "make", "type", "model")]
outs = build_cascade(cg, "os", "random", "make", "type", "model")
cmodel = helper.make_model(helper.make_graph(
    cg.nodes + [helper.make_node("Concat", list(outs), ["out"], axis=1)], "cascade", cin,
    [helper.make_tensor_value_info("out", TP.INT64, [None, 3])], cg.inits),
    opset_imports=[helper.make_opsetid("", ONNX_OPSET)])
cmodel.ir_version = IR_VERSION
combos = np.array(list(itertools.product(range(len(OS_LABELS)), (0, 1), range(len(MAKE_L)), range(len(TYPE_L)),
                                         range(len(MODEL_L)))), dtype=np.int64)
got = ort.InferenceSession(cmodel.SerializeToString(), providers=["CPUExecutionProvider"]).run(None, {
    "os": combos[:, [0]], "random": combos[:, [1]].astype(bool), "make": combos[:, [2]],
    "type": combos[:, [3]], "model": combos[:, [4]]})[0]
bad = sum(cascade(OS_LABELS[o], r, MAKE_L[mk], TYPE_L[ty], MODEL_L[mo])[1:4]
          != (MAKE_L[g_[0]], TYPE_L[g_[1]], MODEL_L[g_[2]]) for (o, r, mk, ty, mo), g_ in zip(combos, got))
print(f"cascade: {len(combos)} tổ hợp, lệch {bad}")
assert bad == 0

cascade: 7560 tổ hợp, lệch 0


In [23]:
# 2. Đầu cuối


def os_probe(base):
    hostnames = ["iPhone", " iPhone ", "IPHONE-cua-Lan", "iphoneđ", "iphone-đ", "ipad-pro", "ipod", "MacBook-Air",
                 "imac", "mac-mini", "macpro", "android-0123456789abcdef", "Galaxy-A51", "sm-a515f", "redmi-9",
                 "xiaomi", "poco-x3", "mi-9", "realme-6i", "oppo-a92", "vivo-1906", "oneplus", "pixel-7",
                 "huawei-p30", "honor-x8", "nokia", "moto-g", "DESKTOP-AB12CD3", "desktop-ducanh", " Desktop-AB12 ",
                 "laptop-vaf70sq6", "win-abcd1234", "WIN-AB", "raspberrypi", "raspberrypi4", "ubuntu", "debian-x",
                 "fedora", "archlinux", "ESP_0C3994", "tuya-plug", "shelly1", "tasmota", "wled-1", "linova",
                 "Johns-iPhone", "ĐIỆN-THOẠI", "none", " NaN ", "", MISSING]
    vcis = ["MSFT 5.0", " msft 5.0", "MSFT5.0", "msft ", "MSFT \t", "android-dhcp-11", "ANDROID-DHCP-16",
            "android-dhcp", "dhcpcd-6.8.2:Linux-4.4.22+:armv7l", "udhcp 1.26.2", MISSING]
    prls = []
    for key in os_bundle["signatures"]:
        opts = [str(v) for v in sorted(key)]
        prls += [",".join(opts), ",".join(reversed(opts)), ",".join(opts[:-1]), ",".join(opts + ["162"]),
                 ",".join([opts[0]] + opts)]
    mdns = ["_companion link _tcp local", "_rdlink _tcp local", "x _companion link", "_airplay _tcp local", MISSING]
    ouis = [MISSING, "00:e0:4c", "02:e0:4c", "00:11:22", "AA:BB:CC", "fe:db:aa"]   # thiếu, thật, ngẫu nhiên...
    rows = []
    for b, o in itertools.product(base, ouis):
        rows += [dict(b, dhcp_hostname=h, mac_oui=o) for h in hostnames]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=v, mac_oui=o) for v in vcis]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=MISSING, dhcp_prl=p, mac_oui=o) for p in prls]
        rows += [dict(b, dhcp_hostname=MISSING, dhcp_vci=MISSING, dhcp_prl=MISSING, mdns_tokens=m, mac_oui=o)
                 for m in mdns]
    return np.asarray([[str(r[c]) for c in COLS] for r in rows], dtype=object)


def reference(m):
    iden = base_sess.run([OUT_LABELS], {IN_FEATURES: m})[0].astype(str)
    frame = matrix_frame(m, COLS, IDEN_NUM)
    frame["mac_is_random"] = [mac_random_from_oui(v) for v in frame["mac_oui"]]
    decided = os_decide(os_bundle, frame)
    out = [cascade(d[0], r, *row)[1:4] for d, r, row in zip(decided, frame["mac_is_random"], iden)]
    return np.asarray(out, dtype=str).reshape(-1, 3), iden


base_sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
new_sess = ort.InferenceSession(model.SerializeToString(), providers=["CPUExecutionProvider"])
capture = capture_frame()
cap_m = to_input_matrix(capture, COLS)
checks = {"capture": cap_m, "ép OS": os_probe(capture.drop_duplicates("mac").head(5)[COLS].to_dict("records"))}
rows = []
for name, m in checks.items():
    want, iden = reference(m)
    got = new_sess.run([OUT_LABELS], {IN_FEATURES: m})[0].astype(str)
    rows.append({"nguồn": name, "dòng": len(m), **{f"{h} lệch": int((want[:, i] != got[:, i]).sum())
                                                   for i, h in enumerate(HEADS)},
                 "ô đổi bởi cascade": int((want != iden).sum())})
check = pd.DataFrame(rows).set_index("nguồn")
display(check)
assert check.filter(like=" lệch").to_numpy().sum() == 0
print("OK — file mới trả đúng cascade(os_decide(), sdc_iden.onnx)")

sample = pd.DataFrame(new_sess.run([OUT_LABELS], {IN_FEATURES: cap_m})[0], columns=HEADS)
sample.insert(0, "dhcp_hostname", cap_m[:, COLS.index("dhcp_hostname")])
sample.insert(1, "iden gốc", ["/".join(r) for r in reference(cap_m)[1]])
display(sample.drop_duplicates().head(20))

,dòng,make lệch,type lệch,model lệch,ô đổi bởi cascade
nguồn,,,,,
capture,62,0,0,0,3
ép OS,5010,0,0,0,3215


OK — file mới trả đúng cascade(os_decide(), sdc_iden.onnx)


,dhcp_hostname,iden gốc,make,type,model
0,lee_kingdom,Generic Laptop/Laptop/Windows Desktop DELL,Generic Laptop,Laptop,Windows Desktop DELL
1,iphone,Apple/Smartphone/iPhone,Apple,Smartphone,iPhone
2,linova,Linova/Linux/Laptop/Linova Laptop HP,Linova/Linux,Laptop,Linova Laptop HP
3,desktop-ducanh,Generic Laptop/Laptop/Windows Desktop DELL,Generic Laptop,Laptop,Windows Desktop DELL
4,<missing>,Generic Laptop/Laptop/Windows Laptop HP,Generic Laptop,Laptop,Windows Laptop HP
5,desktop-dngjhrt,Generic Laptop/Laptop/Windows Desktop HP,Generic Laptop,Laptop,Windows Desktop HP
6,<missing>,Camera/IP Camera/Generic IP Camera,Camera,IP Camera,Generic IP Camera
7,<missing>,Xiaomi/Smartphone/Redmi Note 10,Xiaomi,Smartphone,Redmi Note 10
8,samsung-cua-duc-anh,Samsung/Smartphone/Samsung Galaxy,Samsung,Smartphone,Samsung Galaxy
9,raspberrypi,Raspberry Pi/Single-board Computer/Raspberry Pi,Raspberry Pi,Single-board Computer,Raspberry Pi


## 5. Thử trên thiết bị chưa thấy, và kiểm hồi quy

Chuyển từ 05b (mục 7 cũ), nhưng giờ cột "sau" là output của **chính file `sdc_os.onnx`** vừa dựng, không phải
code Python. "trước" là `sdc_iden.onnx` (09); `os [nguồn]` và ghi chú lấy từ `os_decide` / `cascade` để đọc lý do.

* 5a — pcap tự thu: 3 MAC không có trong bảng nhãn (2 hostname `iphone` + 1 MAC Fingerbank nói Apple), có và bỏ hostname;
* 5b — `test_pcap_collection`: thiết bị đã biết, MAC mới, ngày khác;
* 5c — iPhone trong log đối tác (PRL bị sort, có option 162, không TLS) và `realme-6i` (chỉ hostname / gói DHCP giả lập);
* 5d — hồi quy: cascade không được làm hỏng câu đúng, không được biến thiết bị IoT thành điện thoại/PC.

In [24]:
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)


def prep(frame):
    """Khung CSV/aggregate -> đủ cột cho contract 12 cột và head OS."""
    f = frame.copy()
    f["mac_is_random"] = f["mac"].map(mac_is_random)
    f["mac_oui"] = f["mac"].map(mac_oui)
    for c in ("dhcp_hostname", "mdns_model"):
        f[c] = f[c].fillna(MISSING).astype(str) if c in f.columns else MISSING
    for c in set(os_bundle["cols"]) | set(COLS):
        numeric = c.startswith(("dhcp_opt_", "has_")) or c == "dhcp_prl_len"
        if c not in f.columns:
            f[c] = 0 if numeric else MISSING
        else:  # dòng thiếu nguồn (vd. mới có hostname) -> ô NaN
            f[c] = f[c].fillna(0).astype(int) if numeric else f[c].fillna(MISSING)
    return f


def release_out(m):
    return new_sess.run([OUT_LABELS], {IN_FEATURES: m})[0].astype(str)


def explain(frame, use_hostname=True):
    f = prep(frame)
    if not use_hostname:
        f["dhcp_hostname"] = MISSING
    m = to_input_matrix(f, COLS)
    old = base_sess.run([OUT_LABELS], {IN_FEATURES: m})[0].astype(str)
    new = release_out(m)
    rows = []
    for i, (label, src, _) in enumerate(os_decide(os_bundle, f)):
        *py, note = cascade(label, f["mac_is_random"].iat[i], *old[i])[1:]
        assert tuple(py) == tuple(new[i]), (py, new[i])          # file release == đường Python
        rows.append({"mac": f["mac"].iat[i], "hostname": f["dhcp_hostname"].iat[i],
                     "trước (sdc_iden)": "/".join(old[i]), "os": f"{label} [{src}]",
                     "sau (sdc_os.onnx)": "/".join(new[i]), "ghi chú": note})
    return pd.DataFrame(rows)


sessions, feature_groups = load_sessions(SESSIONS_PATH)
is_field = sessions["scenario"].eq("FIELD")

In [25]:
# 5a. pcap tự thu: MAC không có trong bảng nhãn
cap = pd.read_csv(FEAT_DIR / "session_features_capture.csv")
dhcp = pd.read_csv(FEAT_DIR / "dhcp_features_capture.csv")
cap["dhcp_hostname"] = cap["mac"].map(dhcp.dropna(subset=["hostname"]).drop_duplicates("mac")
                                      .set_index("mac")["hostname"])
labelled_macs = set(sessions.loc[is_field, "mac"])
unseen = cap[~cap["mac"].isin(labelled_macs)]
for use_hostname in (True, False):
    print(f"--- {unseen['mac'].nunique()} MAC chưa thấy, {'có' if use_hostname else 'BỎ'} hostname")
    display(explain(unseen, use_hostname))

--- 3 MAC chưa thấy, có hostname


,mac,hostname,trước (sdc_iden),os,sau (sdc_os.onnx),ghi chú
0,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
1,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
2,06:27:04:ee:2a:40,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
3,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
4,06:27:04:ee:2a:40,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
5,06:27:04:ee:2a:40,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
6,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
7,6e:4c:77:fc:06:d3,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~1.00],Apple/Smartphone/__unknown__,make suy từ OS
8,6e:4c:77:fc:06:d3,<missing>,__unknown__/__unknown__/__unknown__,iOS [mdns],Apple/Smartphone/__unknown__,type suy từ OS; make suy từ OS
9,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,


--- 3 MAC chưa thấy, BỎ hostname


,mac,hostname,trước (sdc_iden),os,sau (sdc_os.onnx),ghi chú
0,16:76:ae:9b:f1:a0,<missing>,__unknown__/__unknown__/__unknown__,__unknown__ [abstain],__unknown__/__unknown__/__unknown__,
1,16:76:ae:9b:f1:a0,<missing>,Apple/Smartphone/iPhone,iOS [mdns],Apple/Smartphone/iPhone,
2,06:27:04:ee:2a:40,<missing>,__unknown__/Smartphone/__unknown__,iOS [mdns],Apple/Smartphone/__unknown__,make suy từ OS
3,16:76:ae:9b:f1:a0,<missing>,__unknown__/__unknown__/__unknown__,__unknown__ [abstain],__unknown__/__unknown__/__unknown__,
4,06:27:04:ee:2a:40,<missing>,__unknown__/__unknown__/__unknown__,iOS [mdns],Apple/Smartphone/__unknown__,type suy từ OS; make suy từ OS
5,06:27:04:ee:2a:40,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~1.00],Apple/Smartphone/__unknown__,make suy từ OS
6,16:76:ae:9b:f1:a0,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~0.90],Apple/Smartphone/__unknown__,make suy từ OS
7,6e:4c:77:fc:06:d3,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~1.00],Apple/Smartphone/__unknown__,make suy từ OS
8,6e:4c:77:fc:06:d3,<missing>,__unknown__/__unknown__/__unknown__,iOS [mdns],Apple/Smartphone/__unknown__,type suy từ OS; make suy từ OS
9,16:76:ae:9b:f1:a0,<missing>,__unknown__/__unknown__/__unknown__,__unknown__ [abstain],__unknown__/__unknown__/__unknown__,


In [26]:
# 5b. test_pcap_collection: thiết bị đã biết, MAC mới, ngày khác
test3 = pd.read_csv(FEAT_DIR / "test_pcap_collection" / "session_features.csv")
test3["dhcp_hostname"] = test3["device"]
for use_hostname in (False, True):
    print(f"--- test_pcap_collection, {'có' if use_hostname else 'BỎ'} hostname")
    display(explain(test3, use_hostname))

--- test_pcap_collection, BỎ hostname


,mac,hostname,trước (sdc_iden),os,sau (sdc_os.onnx),ghi chú
0,e2:1c:d2:b0:af:a5,<missing>,__unknown__/Smartphone/__unknown__,Android [vci],__unknown__/Smartphone/__unknown__,
1,7e:f9:e6:c0:a1:aa,<missing>,__unknown__/Smartphone/__unknown__,Android [vci],__unknown__/Smartphone/__unknown__,
2,6e:18:8d:88:08:15,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~1.00],Apple/Smartphone/__unknown__,make suy từ OS


--- test_pcap_collection, có hostname


,mac,hostname,trước (sdc_iden),os,sau (sdc_os.onnx),ghi chú
0,e2:1c:d2:b0:af:a5,Redmi-Note-10,Xiaomi/Smartphone/Redmi Note 10,Android [hostname],Xiaomi/Smartphone/Redmi Note 10,
1,7e:f9:e6:c0:a1:aa,Samsung-cua-Duc-Anh,Samsung/Smartphone/__unknown__,Android [vci],Samsung/Smartphone/__unknown__,
2,6e:18:8d:88:08:15,iPhone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,


In [27]:
# 5c. iPhone trong log đối tác: PRL bị sort, có option 162 chưa từng thấy, không TLS.
#     realme-6i trong log router: mới có MAC + hostname; thêm một bản GIẢ LẬP gói DHCP Android.
fc = feature_groups["all"]
partner = aggregate([{"proto": "dhcp", "opt55": "1,108,114,119,121,15,162,252,3,6"}]
                    + [{"proto": "dns", "qname": q} for q in ("captive.apple.com", "gateway.icloud.com")]
                    + [{"proto": "mdns", "qname": q} for q in ("_companion-link._tcp.local", "_rdlink")], fc)
realme_sim = aggregate([{"proto": "dhcp", "opt55": "1,3,6,15,26,28,51,58,59,43,114,108",
                         "opt60": "android-dhcp-11"}], fc)
demo = pd.DataFrame([
    dict(partner, mac="fa:4c:89:0d:da:5f", dhcp_hostname=None),
    {"mac": "22:5e:21:5d:d4:53", "dhcp_hostname": "realme-6i"},
    dict(realme_sim, mac="22:5e:21:5d:d4:53", dhcp_hostname=None),
], index=["iPhone đối tác, không hostname", "realme-6i, chỉ hostname", "realme-6i GIẢ LẬP, không hostname"])
display(explain(demo).set_index(demo.index))

,mac,hostname,trước (sdc_iden),os,sau (sdc_os.onnx),ghi chú
"iPhone đối tác, không hostname",fa:4c:89:0d:da:5f,<missing>,__unknown__/__unknown__/__unknown__,iOS [prl~0.90],Apple/Smartphone/__unknown__,type suy từ OS; make suy từ OS
"realme-6i, chỉ hostname",22:5e:21:5d:d4:53,realme-6i,__unknown__/__unknown__/__unknown__,Android [hostname],__unknown__/Smartphone/__unknown__,type suy từ OS
"realme-6i GIẢ LẬP, không hostname",22:5e:21:5d:d4:53,<missing>,__unknown__/__unknown__/__unknown__,Android [vci],__unknown__/Smartphone/__unknown__,type suy từ OS


In [28]:
# 5d. Hồi quy trên chính file release
labelled = prep(cap[cap["mac"].isin(labelled_macs)].merge(
    sessions.loc[is_field, ["mac", "make", "type", "model"]].drop_duplicates("mac"), on="mac"))
m = to_input_matrix(labelled, COLS)
old, new = base_sess.run([OUT_LABELS], {IN_FEATURES: m})[0].astype(str), release_out(m)
truth = labelled[["make", "type", "model"]].to_numpy(dtype=str)
broken = int(((old == truth).all(axis=1) & (new != truth).any(axis=1)).sum())
print(f"capture có nhãn: {len(labelled)} cửa sổ, cascade làm hỏng {broken} câu đúng")

iot = prep(sessions[~is_field])
new = release_out(to_input_matrix(iot, COLS))
leaked = int(np.isin(new[:, HEADS.index("type")], sorted(GENERAL_PURPOSE_TYPES)).sum())
print(f"CIC-2022 (in-sample): {leaked}/{len(iot)} cửa sổ IoT bị gán type đa dụng")
assert broken == 0 and leaked == 0

capture có nhãn: 50 cửa sổ, cascade làm hỏng 0 câu đúng
CIC-2022 (in-sample): 0/8139 cửa sổ IoT bị gán type đa dụng


## 6. Lưu bản release

Contract của 09 giữ nguyên các khoá chung (cột, nhãn, hash, hierarchy, ngưỡng iden), thêm `os_head` (luật,
chữ ký, cascade, quy ước `mac_is_random`) và nguồn. `manifest.json` là chỗ duy nhất trả lời "bản deploy này
ghép từ run nào": 13 đọc nó để biết dựng TFLite từ đâu.

In [29]:
RELEASE_ID = datetime.now().strftime("%Y%m%d_%H%M%S_release")
release_dir = run_dir / "release"
contract = {k: v for k, v in base_contract.items() if k not in ("size_optimization",)}
contract.update({
    "format": "sdc-iden-os-onnx-v1", "run_id": RELEASE_ID, "created": datetime.now().isoformat(timespec="seconds"),
    "source_iden_onnx": {"run_id": base_contract["run_id"], "sha256": hashlib.sha256(onnx_path.read_bytes()).hexdigest()},
    "output_note": "make/type/model sau cascade OS -> type -> make -> model (05b)",
    "os_head": {
        "run_id": run_dir.name, "classes": OS_LABELS, "forest_threshold": float(os_bundle["threshold"]),
        "mac_is_random": "từ cột mac_oui: <missing>/rỗng = MAC ngẫu nhiên hoá (router luôn biết MAC, extractor "
                         "chỉ trả <missing> khi MAC ngẫu nhiên); OUI gửi nguyên thì xét bit U/L 0x02 của octet đầu",
        "l0_hostname": [{"search": p, "os": o} for p, o in HOST_OS],
        "l0_vci": [{"prefix": "msft ", "os": "Windows"},
                   {"prefix": "android-dhcp-", "os": "Android", "require_mac_random": True}],
        "l1_prl": {"min_jaccard": MIN_JACCARD, "prl_format": "dạng chuẩn của contract (như 09)",
                   "signatures": [{"options": sorted(int(v) for v in k), "os": sorted(o)}
                                  for k, o in os_bundle["signatures"].items()]},
        "l1_mdns": {"contains_any": ["_companion link", "_rdlink"], "os": "iOS", "require_mac_random": True},
        "forest_gate": "lớp khác Embedded thì cửa sổ phải có DHCP hoặc mDNS",
        "cascade": {"type_ok": {k: sorted(v) for k, v in TYPE_OK.items()},
                    "general_purpose_types": sorted(GENERAL_PURPOSE_TYPES),
                    "make_ok": {k: sorted(v) for k, v in MAKE_OK.items()},
                    "fill": {"iOS": "Smartphone + Apple", "Android + MAC ngẫu nhiên": "Smartphone",
                             "Windows": "Laptop", "macOS": "Laptop + Apple"},
                    "model": "make hoặc type là __unknown__ thì model cũng __unknown__"},
    },
})
del model.metadata_props[:]
for key, value in (("sdc.contract", json.dumps(contract, ensure_ascii=False)),
                   ("sdc.columns_sha256", contract["columns_sha256"]),
                   ("sdc.labels_sha256", contract["labels_sha256"]),
                   ("sdc.run_id", RELEASE_ID), ("sdc.format", contract["format"])):
    model.metadata_props.append(onnx.StringStringEntryProto(key=key, value=value))

if release_dir.exists():   # chạy lại 12 trên cùng run: bỏ cả release cũ, kể cả TFLite 13 dựng từ ONNX cũ
    shutil.rmtree(release_dir)
release_dir.mkdir()
payload = model.SerializeToString()
(release_dir / RELEASE_ONNX).write_bytes(payload)
(release_dir / "sdc_os_contract.json").write_text(json.dumps(contract, ensure_ascii=False, indent=1),
                                                  encoding="utf-8")
manifest = {
    "format": "sdc-release-v1", "release_id": RELEASE_ID, "created": contract["created"],
    "sources": {"run": run_dir.name, "iden_run_id": base_contract["run_id"],
                "iden_onnx_sha256": contract["source_iden_onnx"]["sha256"]},
    "files": {RELEASE_ONNX: {
        "made_by": "12_build_release_onnx", "runtime": "onnxruntime", "contract_format": contract["format"],
        "sha256": hashlib.sha256(payload).hexdigest(), "bytes": len(payload),
        "check": check.reset_index().to_dict("records"), "cascade_combos_checked": int(len(combos)),
        "regression": {"capture_broken": broken, "iot_general_type": leaked},
    }},
}
(release_dir / RELEASE_MANIFEST).write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=int),
                                           encoding="utf-8")
for p in sorted(release_dir.iterdir()):
    print(f"{p.stat().st_size / 1024:9.1f} KiB  {p}")

      1.1 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\manifest.json
   2284.5 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\sdc_os.onnx
     12.6 KiB  D:\01.AI_Security\02.SDC\Models\20261005_140727_release\sdc_os_contract.json
